# Baselines (issue #17)

Synthèse des quatre prévisions de référence que tout modèle doit battre : scores du backtest, couverture des fenêtres, puis prédiction contre réalité sur la semaine du 11 novembre 2025.

- Code : `src/traffic_prediction/models/baselines.py`. Choix et limites : `docs/methodology.md`, section « Baselines (issue #17) ».
- Les scores sont relus dans `docs/experiments.csv` (dernier run de chaque baseline) : le backtest n'est pas relancé ici.
- La partie graphique lit les données locales. Prérequis : `python -m traffic_prediction.data.update`, puis `python -m traffic_prediction.data.build --calendar`.
- Installer le projet avec `python -m pip install -e .` avant d'importer le package.

In [ ]:
import json

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from traffic_prediction.data.query import query_traffic
from traffic_prediction.data.traffic import REPO_ROOT
from traffic_prediction.evaluate import DEFAULT_EXPERIMENTS, target_window
from traffic_prediction.models.baselines import (
    HolidayAwarePreviousWeekBaseline, PreviousWeekBaseline, load_calendar,
)

BASELINES = ["PreviousWeekBaseline", "FourWeekMeanBaseline", "DayTypeProfileBaseline",
             "HolidayAwarePreviousWeekBaseline"]
pd.set_option("display.precision", 2)

## 1. Scores consignés

Une ligne de `experiments.csv` par run ; on garde le dernier run de chaque baseline. Les scores sont des RMSE par fenêtre, arc, variable et sous-ensemble (`all` : toutes les heures ; `atypical` : samedi, dimanche et jours fériés). Seules les heures dont la vérité est `ok` sont notées.

In [ ]:
log = pd.read_csv(DEFAULT_EXPERIMENTS)
runs = log[log["model"].isin(BASELINES)].groupby("model").tail(1).set_index("model")
scores = pd.concat(
    [pd.DataFrame(json.loads(runs.loc[name, "scores"])["scores"]).assign(model=name.removesuffix("Baseline"))
     for name in BASELINES],
    ignore_index=True,
)
scores["cutoff"] = pd.to_datetime(scores["cutoff"]).dt.date
runs.loc[BASELINES, ["date", "author"]]

### Couverture : heures notées par fenêtre et par arc

Identique pour les quatre modèles et pour les deux variables. Saint-Pères n'a de vérité que sur quatre fenêtres, Convention n'a qu'une heure le 17 avril 2025, et la fenêtre d'octobre 2026 n'a plus de mesure des Champs-Elysées.

In [ ]:
flow = scores.query("model == 'PreviousWeek' and variable == 'debit_horaire' and subset == 'all'")
coverage = flow.pivot_table(index="cutoff", columns="arc", values="n_scored")
coverage.loc["notées"] = flow.groupby("arc")["n_scored"].sum()
coverage.loc["ignorées"] = flow.groupby("arc")["n_ignored"].sum()
coverage.astype(int)

### RMSE par arc

Moyenne des RMSE des fenêtres notées (débit en véh/h, occupation en points de %), comme le `per_arc` du backtest.

In [ ]:
per_arc = (scores.groupby(["arc", "model", "variable", "subset"], sort=False)["rmse"].mean()
           .unstack(["variable", "subset"]))
per_arc

### Synthèse

- **Moyenne** : moyenne des RMSE par arc et fenêtre, l'agrégat de l'issue #15.
- **Poolée** : RMSE sur toutes les heures notées, recalculée à partir de `rmse` et `n_scored`. Plus proche de la note officielle (une RMSE sur les 360 lignes), elle ne donne pas le même poids à une fenêtre d'une heure qu'à une fenêtre de 120 heures.

Dans le rendu réel, les Champs-Elysées n'ont plus de mesure : les quatre modèles y prévoient le même profil par type de jour. Le choix du modèle ne change donc que Convention et Saint-Pères, d'où la seconde portée.

In [ ]:
def summarize(frame: pd.DataFrame) -> pd.Series:
    """Mean of window RMSEs and RMSE pooled over all scored hours."""
    scored = frame[frame["n_scored"] > 0]
    pooled = np.sqrt((scored["n_scored"] * scored["rmse"] ** 2).sum() / scored["n_scored"].sum())
    return pd.Series({"moyenne": scored["rmse"].mean(), "poolée": pooled})


scopes = {"3 arcs": scores, "Convention + Saint-Pères": scores[scores["arc"].isin(["Convention", "Saint-Pères"])]}
summary = pd.concat(
    {scope: frame.query("subset == 'all'").groupby(["model", "variable"], sort=False)[["rmse", "n_scored"]]
     .apply(summarize).unstack("variable") for scope, frame in scopes.items()},
    axis=1,
)
summary

## 2. Prédiction contre réalité : semaine du 11 novembre 2025

Coupure le jeudi 6 novembre 2025 à 23:00 ; fenêtre du samedi 8 au mercredi 12 novembre, avec le mardi 11 novembre férié. On compare la semaine précédente, qui copie le mardi 4 novembre, à la variante qui prend le dernier dimanche connu (2 novembre) pour un jour férié. Hors jour férié, les deux baselines sont identiques : la courbe orange y recouvre la bleue. Les heures sont des étiquettes de fin d'heure, comme dans les données ; les trous de la réalité sont des heures sans mesure `ok`.

In [ ]:
CUTOFF = pd.Timestamp("2025-11-06")
ARCS = ["Convention", "Saint-Pères"]
times = target_window(CUTOFF)
data = query_traffic("SELECT * FROM traffic WHERE datetime <= ?", [times[-1].to_pydatetime()])
history = data[data["datetime"] < CUTOFF + pd.Timedelta(days=1)]
calendar = load_calendar()
models = {
    "Semaine précédente": PreviousWeekBaseline(calendar),
    "Semaine précédente, férié = dernier dimanche": HolidayAwarePreviousWeekBaseline(calendar),
}
forecasts = {label: model.predict(history, times).set_index(["arc", "datetime"]) for label, model in models.items()}
truth = data[data["qualite_mesure"].eq("ok") & data["datetime"].isin(times)].set_index(["arc", "datetime"])
# Rule used for each forecast hour (see "sources_" in baselines.py).
pd.DataFrame({label: model.sources_.groupby(level="arc").value_counts() for label, model in models.items()}).fillna(0).astype(int)

In [ ]:
INK, MUTED, GRID, BAND = "#0b0b0b", "#52514e", "#e4e3df", "#eeede9"
COLORS = dict(zip(models, ["#2a78d6", "#eb6834"]))  # Reference palette, slots 1 and 2.
UNITS = {"debit_horaire": "débit (véh/h)", "taux_occupation": "occupation (%)"}
DAYS = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
holiday = pd.Timestamp("2025-11-11")

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for row, arc in enumerate(ARCS):
    for col, variable in enumerate(UNITS):
        ax = axes[row, col]
        ax.axvspan(holiday, holiday + pd.Timedelta(days=1), color=BAND, zorder=0)
        ax.text(holiday + pd.Timedelta(hours=12), 0.98, "11/11 férié", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9, color=MUTED)
        ax.plot(times, truth.loc[arc, variable].reindex(times), color=INK, linewidth=1.5, label="Réalité")
        for label, forecast in forecasts.items():
            ax.plot(times, forecast.loc[arc, variable], color=COLORS[label], linewidth=1.5, label=label)
        ax.set_title(f"{arc}, {UNITS[variable]}", loc="left", fontsize=11, color=INK)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.12)  # Headroom for the holiday label.
        ax.grid(axis="y", color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=9)
for ax in axes[1]:
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: f"{DAYS[mdates.num2date(x).weekday()]} {mdates.num2date(x):%d/%m}"))
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", ncol=3, frameon=False, bbox_to_anchor=(0.01, 0.955), fontsize=10)
fig.suptitle("Le jour férié ressemble à un dimanche : copier le mardi précédent surestime ses pointes",
             x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(REPO_ROOT / "outputs" / "figures" / "baseline_holiday_week_2025-11.png", dpi=150, bbox_inches="tight")
plt.show()

RMSE de cette fenêtre, d'après le journal (toutes les heures notées) :

In [ ]:
(scores.query("cutoff == @CUTOFF.date() and arc in @ARCS and subset == 'all'")
 .pivot_table(index="model", columns=["variable", "arc"], values="rmse", sort=False))

## Conclusions

- **Baseline retenue pour le rendu : `HolidayAwarePreviousWeekBaseline`.** Sur Convention et Saint-Pères, les seuls arcs où le choix du modèle change le rendu, elle est la meilleure des quatre pour le débit et l'occupation, sur toutes les heures comme sur les jours atypiques.
- **Point de départ de la présentation** (run du 31 octobre 2026, RMSE poolée sur Convention et Saint-Pères, toutes heures) : environ **97 véh/h** pour le débit et **2,0 points** pour l'occupation.
- Sur les trois arcs, `DayTypeProfileBaseline` est la meilleure grâce aux Champs-Elysées, dont la semaine précédente est très bruitée.

**Limites.**

- **Champs-Elysées** : aucune mesure après le 19 novembre 2025. Le rendu y utilisera un profil vieux de près d'un an ; son score de backtest, appris jusqu'à chaque coupure, est optimiste.
- **Couverture inégale** : les moyennes par arc ne portent pas sur les mêmes semaines ; toujours lire les RMSE avec les heures notées.
- **11 novembre 2026** : un mercredi férié ; la variante retenue copie le dimanche 1er novembre, lui-même férié (Toussaint) et en vacances scolaires.
- **Convention horaire** : les étiquettes sont des fins d'heure ; la convention attendue par BCG X reste à confirmer pour le rendu (issue #18).